# 🔬 Arm 4 (Priority 4 — Process Verification): Stepwise Execution-Gated RLVR (Step-RLVR)
**Project:** Reduction Ladder for Code & Multi-Arm Mitigation  
**Organization:** Orange Innovation Labs  
**Literature:** CodePRM (ACL 2025) · ExecVerify (ICSE 2026)  

---

## 🎯 Core Problem
On complex algorithmic levels (L4: Difficult, L5: Combine), binary rewards are **extremely sparse**:
- A model correctly solves 4/5 sub-goals but misses a boundary condition → $R = 0$, **discarding all useful signal!**

**Step-RLVR** introduces **Process Reward Models (PRMs)**: intermediate contract checking of sub-functions.

### Stepwise Reward Formulation:
$$\mathcal{R}_{\text{stepwise}}(y) = \sum_{s=1}^{S} w_s \cdot \mathbb{I}(\text{Contract}_s(y_{1:s}) = \text{Valid})$$

| Property | Binary RLVR | Step-RLVR (Arm 4) |
|---|---|---|
| Reward granularity | Terminal only ($R \in \{0,1\}$) | Stepwise partial credits |
| Signal density | Sparse (L4/L5 near-zero) | Dense at every sub-goal |
| Annotation cost | None | Step contracts (moderate) |

> See Table 1 §3.4 (CodePRM [11, 12]) for cross-paper comparison context.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("."))

from src.infrastructure.sandbox import SubprocessSandbox
sandbox = SubprocessSandbox(default_timeout=3.0)
print("✅ Arm 4 (Step-RLVR) Environment Initialized.")

---
## 1. Stepwise Contract Verifier Engine

In [ ]:
# Canonical implementation imported from src.arms.arm4_step_rlvr
from src.arms.arm4_step_rlvr import StepContract, StepwiseContractVerifier, StepwiseRewardEngine

verifier = StepwiseContractVerifier(sandbox)
print("✅ Canonical StepwiseContractVerifier and StepContract imported from src.arms.arm4_step_rlvr.")


---
## 2. Simulation: Partial Success on a Multi-Stage L4/L5 Task

**Task:** Parse raw data string → Compute median → Format report string  
**Candidate code:** Solves Steps 1 & 2 correctly but has a bug in Step 3.

In [ ]:
PROMPT = "Implement a data processing pipeline: parse a comma-separated string, compute the median, and format the result."

# Candidate code: Step 3 has a bug (crashes on empty string)
partial_code = """
def parse_stream(raw: str):
    return [int(x.strip()) for x in raw.split(',') if x.strip()]

def compute_median(nums):
    s = sorted(nums)
    n = len(s)
    if n == 0: return 0
    mid = n // 2
    return s[mid] if n % 2 != 0 else (s[mid-1] + s[mid]) / 2

def format_summary(raw: str):
    # Bug: does not handle empty string!
    nums = parse_stream(raw)
    return f"MEDIAN={compute_median(nums)}"
"""

step_contracts = [
    {
        "name": "Step 1: parse_stream",
        "entry_point": "parse_stream",
        "weight": 0.3,
        "test": "assert parse_stream('1, 2, 3') == [1, 2, 3]\nassert parse_stream('5') == [5]",
    },
    {
        "name": "Step 2: compute_median",
        "entry_point": "compute_median",
        "weight": 0.4,
        "test": "assert compute_median([3, 1, 2]) == 2\nassert compute_median([1, 2, 3, 4]) == 2.5",
    },
    {
        "name": "Step 3: format_summary (BUGGY — crashes on empty string)",
        "entry_point": "format_summary",
        "weight": 0.3,
        "test": "assert format_summary('') == 'EMPTY'\nassert format_summary('3,1,2') == 'MEDIAN=2'",
    },
]

result = verifier.evaluate_steps(partial_code, step_contracts, prompt=PROMPT)
df_steps = pd.DataFrame(result["steps"])

print(f"\n📊 Step-RLVR Evaluation:")
print(df_steps.to_string(index=False))
print(f"\n  Binary RLVR Reward   : 0.0  (failed final test → zero credit)")
print(f"  Step-RLVR Reward     : {result['total_stepwise_reward']:.2f} (earned partial credit for Steps 1 & 2!)")

---
## 3. Reward Density Comparison: Binary vs Step-RLVR

In [ ]:
df_steps['Status'] = df_steps['Passed'].map({True: 'PASS ✅', False: 'FAIL ❌'})

colors = ['#28A745' if p else '#DC3545' for p in df_steps['Passed']]
plt.figure(figsize=(9, 4), dpi=140)
bars = plt.bar(df_steps['Step'], df_steps['Credits'], color=colors, edgecolor='black', linewidth=0.5)
plt.axhline(result['total_stepwise_reward'], color='#FF6400', linestyle='--', linewidth=1.5,
            label=f"Step-RLVR Total: {result['total_stepwise_reward']:.2f}")
plt.axhline(0, color='gray', linestyle=':', linewidth=1, label="Binary RLVR: 0.0")
plt.title('Step-RLVR vs Binary RLVR — Reward Density on L4/L5 Task', fontsize=11, fontweight='bold')
plt.ylabel('Partial Credits Earned', fontsize=10)
plt.ylim(0, 0.55)
plt.xticks(rotation=15, ha='right')
plt.legend(fontsize=9)
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/arm4_step_rlvr_reward.png', dpi=140)
plt.show()
print("✅ Dense process reward preserves useful training signal even on partial solutions!")